# LVIS v1 val: polygons, federated labels, classes matched to PROCTHOR30

[LVIS](https://www.lvisdataset.org/) v1 val: 19,809 COCO 2017 images, 1,203 classes (WordNet synsets, e.g.
`trash_can`, `armchair`), each object a polygon mask. Read with `habitat_embodied_sl/static_datasets/lvis_val.py`'s
`lvis_val_records`, i.e. detectron2's own LVIS loader on Detic's `datasets/lvis/lvis_v1_val.json` and `datasets/coco/`.
Needs `INSTALL.MD`'s Detic data section (`common/vision/setup_detic_data.sh`) run first.

LVIS is *federated*: no image is annotated for every class. Each record carries `neg_category_ids` (classes verified
absent) and `not_exhaustive_category_ids` (classes annotated, but not every instance), both as class indices here.
`lvis_class_mosaic.py` draws the matched classes below on LVIS train.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("BASE_DIR", str(REPO_ROOT))

from common.env_utils.vocab_constants import CLASS_LABELS_PROCTHOR30
from common.vision.detic import matched_classes
from habitat_embodied_sl.static_datasets.detection_sources import showing_records
from habitat_embodied_sl.static_datasets.inspection import matched_table, plot_matched_counts, plot_verification, show_records

VOCAB, THRESHOLD = CLASS_LABELS_PROCTHOR30, 0.9

In [ ]:
from common.vision.detic import LVIS
from habitat_embodied_sl.static_datasets.lvis_val import lvis_val_records

records = lvis_val_records()
print(f"{len(records)} images, {sum(len(r['annotations']) for r in records)} objects, {len(LVIS.classes)} classes")
{key: value for key, value in records[0].items() if key != "annotations"}

## A few examples

Random records, drawn with detectron2's `Visualizer` (masks where the source has them).

In [ ]:
show_records(records, LVIS.classes)

## Classes matched to PROCTHOR30

`common/vision/detic.py`'s `matched_classes`: every LVIS class whose CLIP text embedding is at least `THRESHOLD`
cosine-similar to some `VOCAB` word's, both prompted as `a <name>` -- the classes `lvis_val.py` starts from with
`--vocab` and `--sim-word-thresh`. Change `VOCAB`/`THRESHOLD` above to try another vocabulary.

In [ ]:
matches = matched_classes(LVIS, VOCAB, THRESHOLD)
matched = {index for index, _, _ in matches}
matched_table(LVIS, matches)

## Matched classes' frequency

Object count of each matched class over the images showing at least one (`detection_sources.py`'s `showing_records`,
every one here, as without `--max-images`). `lvis_val.py` keeps the classes with at least `MIN_OCCURRENCES` objects
(`vocabulary.py`, after merging spelling variants), in blue; the grey ones' objects are left out of its test set.

In [ ]:
showing = showing_records(records, matched, None)
plot_matched_counts(showing, LVIS.classes, matched)

## Federated verification

LVIS only verifies a class on an image where it is annotated exhaustively or listed absent (`neg_category_ids`). On
every other image -- annotated but in `not_exhaustive_category_ids`, or neither annotated nor listed -- `lvis_val.py` gives
the class's category a whole-image `iscrowd` box, so COCOeval ignores its unmatched detections there, as LVIS's own
evaluation does: only a detection on an image where the class is listed absent counts as a false positive. A class
unverified on most images is scored almost only on its recall.

In [ ]:
plot_verification(showing, LVIS.classes, matched)

## Building the test set

```bash
PYTHONPATH=. python habitat_embodied_sl/static_datasets/lvis_val.py datasets/lvis_val --vocab <VOCAB words> --sim-word-thresh 0.9
```
writes `datasets/lvis_val/test/test.json` and its images in `datasets/lvis_val/images/`, registered as `lvis_val_test`
(`common/env_utils/dataset_registry.py`) for a training config's `DATASETS.TEST`.